In [ ]:
# Imports and configuration
import os
import math
import requests
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, HTML
from urllib.parse import parse_qs

# Parse query parameters from QUERY_STRING environment variable
query_string = os.environ.get("QUERY_STRING", "")
params = parse_qs(query_string)

# Extract values
entityId = params.get("entity_id", [None])[0]
tenantId = params.get("tenantId", ["DEFAULT"])[0]
timeRange = params.get("timeRange", ["month"])[0]

# Backend base URL (set via CASE_MGMT_BACKEND_URL)
backendUrl = os.getenv('CASE_MGMT_BACKEND_URL', 'http://localhost:3090')

# Shared secret for proxy requests (set JUPYTER_SHARED_SECRET in environment)
JUPYTER_SHARED_SECRET = os.getenv('JUPYTER_SHARED_SECRET')
if JUPYTER_SHARED_SECRET:
    headers = {'x-jupyter-secret': JUPYTER_SHARED_SECRET}
else:
    headers = {}

# Validation
if not entityId:
    display(HTML("""
        <div style="padding:40px;text-align:center;color:#ef4444;">
            <h3>Missing Required Parameter</h3>
            <p>entity_id is required.</p>
            <code>/voila/render/account-network.ipynb?entity_id=xxx&tenantId=yyy&timeRange=month</code>
        </div>
    """))
    raise ValueError("entity_id parameter is required")


In [ ]:
# Fetch entity network analysis data
def fetch_entity_network(entity_id, tenant_id, granularity, backend_url):
    """
    Fetch network data for an entity from the backend.
    
    Args:
        entity_id: The entity ID to fetch network for
        tenant_id: The tenant identifier
        granularity: Time granularity (day, month, year)
        backend_url: Backend API base URL
    
    Returns:
        dict: Network data or None if error
    """
    url = f"{backend_url}/api/v1/jupyter/proxy/network-analysis/entity/{entity_id}"
    params = {
        'tenantId': tenant_id,
        'granularity': granularity
    }
    
    try:
        resp = requests.get(url, params=params, headers=headers, timeout=15)
        resp.raise_for_status()
        return resp.json()
    except requests.exceptions.HTTPError as e:
        print(f'HTTP error fetching entity network: {e}')
        if hasattr(e, 'response') and e.response is not None:
            print(f'Response status: {e.response.status_code}')
            print(f'Response body: {e.response.text[:500]}')
        return None
    except Exception as e:
        print(f'Error fetching entity network: {e}')
        return None

# Fetch data using parameters from URL
data = fetch_entity_network(entityId, tenantId, timeRange, backendUrl)

if not data:
    display(HTML(f"""
        <div style='color:#ef4444; padding: 20px; background: #fee2e2; border: 1px solid #ef4444; border-radius: 8px;'>
            <strong>No network data available</strong><br>
            Entity ID: {entityId}<br>
            Tenant ID: {tenantId}<br>
            Time Range: {timeRange}<br>
            Please ensure the entity exists in the system.
        </div>
    """))


In [ ]:
# Normalize expected network payloads and build node/edge lists
nodes = []
edges = []

if data and isinstance(data, dict):
    network_data = data.get('network')
    if network_data and isinstance(network_data, dict):
        nodes = network_data.get('nodes', [])
        edges = network_data.get('edges', [])
    else:
        nodes = data.get('nodes', [])
        edges = data.get('edges', [])

nodes = [n for n in nodes if isinstance(n, dict)]
edges = [e for e in edges if isinstance(e, dict)]
account_details = data.get('accountDetails', {}) if isinstance(data, dict) else {}

# Get the root node ID (center node) from the network data
root_node_id = None
if data and isinstance(data, dict):
    network_data = data.get('network')
    if network_data and isinstance(network_data, dict):
        root_node_id = network_data.get('rootNodeId')

def build_positions(nodes, root_node_id):
    positions = {}
    n = max(len(nodes), 1)
    center_idx = 0

    for i, node in enumerate(nodes):
        if node.get('id') == root_node_id:
            center_idx = i
            break

    radius = 280
    angle_step = 2 * math.pi / max(n - 1, 1)
    j = 0
    for i, node in enumerate(nodes):
        if i == center_idx:
            positions[node['id']] = (0.0, 0.0)
        else:
            angle = angle_step * j
            positions[node['id']] = (radius * math.cos(angle), radius * math.sin(angle))
            j += 1
    return positions

positions = build_positions(nodes, root_node_id)

# Helper functions to extract flags from the new data structure
def is_node_alerted(node):
    flags = node.get('flags', {})
    return flags.get('alerted', False)

def is_node_investigated(node):
    flags = node.get('flags', {})
    return flags.get('investigated', False)

def is_edge_alerted(edge):
    flags = edge.get('flags', {})
    return flags.get('alerted', False)

def is_edge_investigated(edge):
    flags = edge.get('flags', {})
    return flags.get('investigated', False)

# Helpers
def get_node_status(node, root_node_id):
    if node.get('id') == root_node_id:
        return 'center'
    elif is_node_alerted(node):
        return 'alert'
    elif is_node_investigated(node):
        return 'investigation'
    else:
        return 'normal'

status_map = {n.get('id'): get_node_status(n, root_node_id) for n in nodes if n.get('id')}

def get_node_size(node, root_node_id):
    if node.get('id') == root_node_id:
        return 95
    if is_node_alerted(node):
        return 80
    return 75

size_map = {n.get('id'): get_node_size(n, root_node_id) for n in nodes if n.get('id')}


edge_segments = {
    'alert': {'x': [], 'y': []},
    'investigation': {'x': [], 'y': []},
    'normal': {'x': [], 'y': []}
}

center_id = root_node_id

def offset_edge(x0, y0, x1, y1, start_offset, end_offset):
    dx = x1 - x0
    dy = y1 - y0
    dist = math.hypot(dx, dy)
    if dist == 0:
        return x0, y0, x1, y1
    ux = dx / dist
    uy = dy / dist
    return (
        x0 + ux * start_offset,
        y0 + uy * start_offset,
        x1 - ux * end_offset,
        y1 - uy * end_offset
    )

for e in edges:
    s = e.get('source') or e.get('from')
    t = e.get('target') or e.get('to')
    if s in positions and t in positions:
        x0, y0 = positions[s]
        x1, y1 = positions[t]

        start_offset = (size_map.get(s, 75) / 2) + 6
        end_offset = (size_map.get(t, 75) / 2) + 6
        x0, y0, x1, y1 = offset_edge(x0, y0, x1, y1, start_offset, end_offset)

        # Determine edge color based on edge flags
        if is_edge_alerted(e):
            key = 'alert'
        elif is_edge_investigated(e):
            key = 'investigation'
        else:
            key = 'normal'

        edge_segments[key]['x'] += [x0, x1, None]
        edge_segments[key]['y'] += [y0, y1, None]

edge_traces = [
    go.Scatter(
        x=edge_segments['normal']['x'],
        y=edge_segments['normal']['y'],
        mode='lines',
        line=dict(width=2, color='#8b5cf6', dash='solid'),
        hoverinfo='none',
        showlegend=False
    ),
    go.Scatter(
        x=edge_segments['alert']['x'],
        y=edge_segments['alert']['y'],
        mode='lines',
        line=dict(width=2, color='#ef4444', dash='solid'),
        hoverinfo='none',
        showlegend=False
    ),
    go.Scatter(
        x=edge_segments['investigation']['x'],
        y=edge_segments['investigation']['y'],
        mode='lines',
        line=dict(width=2, color='#f59e0b', dash='solid'),
        hoverinfo='none',
        showlegend=False
    )
]

# Build node traces
node_x = []
node_y = []
node_text = []
node_labels = []
marker_size = []
marker_color = []
marker_line_color = []
marker_line_width = []
node_ids = []

ring_x = []
ring_y = []
ring_sizes = []

for n in nodes:
    nid = n.get('id')
    if nid not in positions:
        continue

    x, y = positions[nid]
    node_x.append(x)
    node_y.append(y)

    label = n.get('label') or ''
    display_id = nid
    if isinstance(nid, str) and not nid.upper().startswith('ACC-') and len(nid) >= 4:
        display_id = f"ACC-{nid[-4:]}"

    # Check if this is the center node first
    is_center = n.get('id') == root_node_id
    
    # For center node, use account holder name if available, otherwise use shortened ID
    if is_center and account_details.get('accountHolder'):
        node_display_name = account_details.get('accountHolder')
        short_label = node_display_name if len(node_display_name) <= 14 else f"{node_display_name[:12]}…"
        node_labels.append(f"{display_id}<br>{short_label}")
        node_text.append(f"<b>{account_details.get('accountHolder')}</b><br>Account ID: {nid}")
    else:
        # For other nodes, just show the shortened account ID since we don't have holder names
        node_labels.append(display_id)
        node_text.append(f"<b>Account</b><br>Account ID: {nid}")

    is_alerted = is_node_alerted(n)
    is_under_investigation = is_node_investigated(n)

    if is_center:
        color = '#ede9fe'
        line_color = '#8b5cf6'
        size = 95
        line_width = 3
    elif is_alerted:
        color = '#fee2e2'
        line_color = '#ef4444'
        size = 80
        line_width = 2
    else:
        color = '#dbeafe'
        line_color = '#3b82f6'
        size = 75
        line_width = 2

    if is_under_investigation:
        ring_x.append(x)
        ring_y.append(y)
        ring_sizes.append(size + 12)

    marker_color.append(color)
    marker_line_color.append(line_color)
    marker_line_width.append(line_width)
    marker_size.append(size)
    node_ids.append(nid)

ring_trace = go.Scatter(
    x=ring_x,
    y=ring_y,
    mode='markers',
    marker=dict(
        size=ring_sizes,
        color='rgba(0,0,0,0)',
        line=dict(width=3, color='#f59e0b')
    ),
    hoverinfo='none',
    showlegend=False
)

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode='markers+text',
    text=node_labels,
    hovertext=node_text,
    hoverinfo='text',
    customdata=node_ids,
    marker=dict(
        color=marker_color,
        size=marker_size,
        line=dict(width=marker_line_width, color=marker_line_color)
    ),
    textposition='middle center',
    textfont=dict(size=9, color='#1e293b', family='Arial, sans-serif'),
    showlegend=False
)

# Calculate summary stats
linked_count = len([n for n in nodes if n.get('id') != root_node_id])
alert_count = len([n for n in nodes if is_node_alerted(n)])

def get_edge_count(e):
    return e.get('txCount') or e.get('transactionCount') or e.get('count') or 1

def get_edge_amount(e):
    return e.get('totalAmount') or e.get('amount') or e.get('totalValue') or 0

def stats_for_node(node_id):
    tx = 0
    total = 0
    for e in edges:
        s = e.get('source') or e.get('from')
        t = e.get('target') or e.get('to')
        if s == node_id or t == node_id:
            tx += get_edge_count(e)
            total += get_edge_amount(e)
    return tx, total

def format_display_id(value):
    if isinstance(value, str) and not value.upper().startswith('ACC-') and len(value) >= 4:
        return f"ACC-{value[-4:]}"
    return value or '-'

initial_node_id = center_id or (node_ids[0] if node_ids else None)
initial_tx, initial_total = stats_for_node(initial_node_id) if initial_node_id else (0, 0)
initial_node = next((n for n in nodes if n.get('id') == initial_node_id), {})
initial_is_center = bool(initial_node.get('id') == root_node_id)

initial_account_id = account_details.get('accountId') if initial_is_center else initial_node.get('id')
initial_holder = (account_details.get('accountHolder') if initial_is_center else initial_node.get('label')) or '-'
initial_relationship = (
    account_details.get('relationship')
    or account_details.get('accountRelationship')
    or account_details.get('relationshipType')
    if initial_is_center else
    (initial_node.get('relationship') or initial_node.get('accountRelationship') or initial_node.get('relationshipType'))
) or '-'
initial_velocity = (account_details.get('velocity') if initial_is_center else initial_node.get('velocity')) or '-'

# Extract alert and investigation status from flags
account_flags = account_details.get('flags', {})
initial_is_alert = account_flags.get('alerted') if initial_is_center else is_node_alerted(initial_node)
initial_is_investigation = account_flags.get('investigated') if initial_is_center else is_node_investigated(initial_node)

fig = go.Figure(
    data=edge_traces + [ring_trace, node_trace],
    layout=go.Layout(
        title=None,
        showlegend=False,
        hovermode='closest',
        dragmode=False,
        margin=dict(b=20, l=20, r=20, t=20),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-380, 380]),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-380, 380]),
        height=560,
        paper_bgcolor='#ffffff',
        plot_bgcolor='#ffffff'
    )
)

if nodes:
    import json
    graph_id = 'account-network-graph'
    graph_html = pio.to_html(fig, full_html=False, config={'displayModeBar': False, 'scrollZoom': False}, div_id=graph_id)

    # Legend
    legend_html = """
    <div style="position: absolute; bottom: 12px; left: 12px; background: white; padding: 10px; border-radius: 8px; border: 1px solid #e2e8f0; box-shadow: 0 4px 10px rgba(15,23,42,0.08); font-family: Arial, sans-serif; font-size: 10px;">
        <div style="font-weight: 600; margin-bottom: 8px; color: #0f172a; font-size: 11px;">Legend</div>
        <div style="display: flex; align-items: center; margin-bottom: 6px;">
            <div style="width: 12px; height: 12px; border-radius: 50%; background: #ede9fe; border: 2px solid #8b5cf6; margin-right: 6px;"></div>
            <span style="color: #475569;">Counterparty</span>
        </div>
        <div style="display: flex; align-items: center; margin-bottom: 6px;">
            <div style="width: 12px; height: 12px; border-radius: 50%; background: #fee2e2; border: 2px solid #ef4444; margin-right: 6px;"></div>
            <span style="color: #475569;">Account with Alert</span>
        </div>
        <div style="display: flex; align-items: center; margin-bottom: 6px;">
            <div style="width: 12px; height: 12px; border-radius: 50%; background: #dbeafe; border: 2px solid #3b82f6; margin-right: 6px;"></div>
            <span style="color: #475569;">Normal Account</span>
        </div>
        <div style="display: flex; align-items: center;">
            <div style="width: 12px; height: 12px; border-radius: 50%; background: #ede9fe; border: 2px solid #8b5cf6; margin-right: 6px; box-shadow: 0 0 0 2px #f59e0b;"></div>
            <span style="color: #475569;">Under Investigation</span>
        </div>
    </div>
    """

    # Zoom Controls
    zoom_controls_html = """
    <div style="position: absolute; top: 16px; right: 16px; background: white; padding: 8px; border-radius: 10px; border: 1px solid #e2e8f0; box-shadow: 0 4px 10px rgba(15,23,42,0.08); display: flex; flex-direction: column; gap: 8px;">
        <button id="zoom-in-btn" style="width: 34px; height: 34px; border: 1px solid #e2e8f0; background: #ffffff; border-radius: 6px; cursor: pointer; display: flex; align-items: center; justify-content: center; color: #1e293b;" onmouseover="this.style.background='#f1f5f9'" onmouseout="this.style.background='#ffffff'">
            <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2">
                <circle cx="11" cy="11" r="8"></circle>
                <line x1="11" y1="8" x2="11" y2="14"></line>
                <line x1="8" y1="11" x2="14" y2="11"></line>
                <line x1="21" y1="21" x2="16.65" y2="16.65"></line>
            </svg>
        </button>
        <button id="zoom-out-btn" style="width: 34px; height: 34px; border: 1px solid #e2e8f0; background: #ffffff; border-radius: 6px; cursor: pointer; display: flex; align-items: center; justify-content: center; color: #1e293b;" onmouseover="this.style.background='#f1f5f9'" onmouseout="this.style.background='#ffffff'">
            <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2">
                <circle cx="11" cy="11" r="8"></circle>
                <line x1="8" y1="11" x2="14" y2="11"></line>
                <line x1="21" y1="21" x2="16.65" y2="16.65"></line>
            </svg>
        </button>
        <button id="reset-zoom-btn" style="width: 34px; height: 34px; border: 1px solid #e2e8f0; background: #ffffff; border-radius: 6px; cursor: pointer; display: flex; align-items: center; justify-content: center; color: #1e293b;" onmouseover="this.style.background='#f1f5f9'" onmouseout="this.style.background='#ffffff'" title="Reset Zoom">
            <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2">
                <path d="M3 12a9 9 0 0 1 9-9 9.75 9.75 0 0 1 6.74 2.74L21 8"></path>
                <path d="M21 3v5h-5"></path>
                <path d="M21 12a9 9 0 0 1-9 9 9.75 9.75 0 0 1-6.74-2.74L3 16"></path>
                <path d="M3 21v-5h5"></path>
            </svg>
        </button>
    </div>
    """

    sidebar_html = f"""
    <div id="sidebar-panel" style="width: 300px; background: white; padding: 24px; font-family: Arial, sans-serif;">
        <div style="margin-bottom: 18px;">
            <div style="font-size: 18px; font-weight: 700; color: #0f172a; margin-bottom: 12px;">Account Details</div>
            <div style="margin-bottom: 10px;">
                <div style="font-size: 10px; color: #64748b; text-transform: uppercase; margin-bottom: 4px;">Account ID</div>
                <div id="sidebar-account-id" style="font-size: 13px; color: #0f172a; font-weight: 600;">{format_display_id(initial_account_id)}</div>
            </div>
            <div style="margin-bottom: 10px;">
                <div style="font-size: 10px; color: #64748b; text-transform: uppercase; margin-bottom: 4px;">Account Holder</div>
                <div id="sidebar-account-holder" style="font-size: 13px; color: #0f172a; font-weight: 600;">{initial_holder}</div>
            </div>
            <div>
                <div style="font-size: 10px; color: #64748b; text-transform: uppercase; margin-bottom: 4px;">Relationship</div>
                <div id="sidebar-relationship" style="font-size: 12px; color: #0f172a; font-weight: 600;">{initial_relationship}</div>
            </div>
        </div>

        <div style="height: 1px; background: #e2e8f0; margin: 16px 0 18px 0;"></div>

        <div id="transaction-stats">
            <div style="font-size: 11px; font-weight: 700; color: #64748b; margin-bottom: 12px; text-transform: uppercase;">Transaction Statistics</div>
            <div style="display: flex; justify-content: space-between; margin-bottom: 8px;">
                <span style="color: #64748b; font-size: 12px;">Transactions:</span>
                <span id="stat-total-tx" style="font-weight: 600; color: #0f172a;">{initial_tx}</span>
            </div>
            <div style="display: flex; justify-content: space-between; margin-bottom: 8px;">
                <span style="color: #64748b; font-size: 12px;">Total Value:</span>
                <span id="stat-total-value" style="font-weight: 600; color: #0f172a;">{initial_total}</span>
            </div>
            <div style="display: flex; justify-content: space-between;">
                <span style="color: #64748b; font-size: 12px;">Velocity:</span>
                <span id="stat-velocity" style="font-weight: 700; color: #ef4444;">{initial_velocity}</span>
            </div>
        </div>

        <div id="alert-box" style="margin-top: 16px; padding: 10px 12px; border: 1px solid #fecaca; background: #fef2f2; color: #dc2626; border-radius: 8px; font-size: 12px; font-weight: 600; display: {'block' if initial_is_alert else 'none'};">
            ⚠ Alert triggered on this account
        </div>

        <div id="investigation-box" style="margin-top: 10px; padding: 10px 12px; border: 1px solid #fde68a; background: #fffbeb; color: #b45309; border-radius: 8px; font-size: 12px; font-weight: 600; display: {'block' if initial_is_investigation else 'none'};">
            ⚠ Currently under investigation
        </div>
    </div>
    """

    interactive_script = f"""
    <script>
    (function() {{
        const nodes = {json.dumps(nodes)};
        const edges = {json.dumps(edges)};
        const originalPositions = {json.dumps(positions)};
        const accountDetails = {json.dumps(account_details)};
        const rootNodeId = {json.dumps(root_node_id)};
        const graphDiv = document.getElementById('{graph_id}');

        let selectedNodeId = {json.dumps(initial_node_id)};
        let currentZoomLevel = 1.0;
        const baseRange = 380;
        const minZoom = 0.3;
        const maxZoom = 3.0;

        const numberFormatter = new Intl.NumberFormat();
        const currencyFormatter = new Intl.NumberFormat(undefined, {{ style: 'currency', currency: 'USD', maximumFractionDigits: 0 }});

        function formatDisplayId(value) {{
            return value || '-';
        }}

        function getEdgeCount(e) {{
            return e.txCount || e.transactionCount || e.count || 1;
        }}

        function getEdgeAmount(e) {{
            return e.totalAmount || e.amount || e.totalValue || 0;
        }}

        function statsForNode(nodeId) {{
            let tx = 0;
            let total = 0;
            edges.forEach(e => {{
                const src = e.source || e.from;
                const tgt = e.target || e.to;
                if (src === nodeId || tgt === nodeId) {{
                    tx += getEdgeCount(e);
                    total += getEdgeAmount(e);
                }}
            }});
            return {{ tx, total }};
        }}

        function getScaledPositions() {{
            const scaledPos = {{}};
            for (const nodeId in originalPositions) {{
                const pos = originalPositions[nodeId];
                scaledPos[nodeId] = [pos[0] * currentZoomLevel, pos[1] * currentZoomLevel];
            }}
            return scaledPos;
        }}

        function getNodeSize(node) {{
            if (node.id === rootNodeId) return 95;
            const flags = node.flags || {{}};
            if (flags.alerted) return 80;
            return 75;
        }}

        function offsetEdge(x0, y0, x1, y1, startOffset, endOffset) {{
            const dx = x1 - x0;
            const dy = y1 - y0;
            const dist = Math.hypot(dx, dy);
            if (dist === 0) return [x0, y0, x1, y1];
            const ux = dx / dist;
            const uy = dy / dist;
            return [
                x0 + ux * startOffset,
                y0 + uy * startOffset,
                x1 - ux * endOffset,
                y1 - uy * endOffset
            ];
        }}

        function rebuildGraph() {{
            const positions = getScaledPositions();
            const sizeMap = {{}};
            const baseSizes = {{}};
            
            // Calculate base and scaled sizes
            nodes.forEach(n => {{
                const baseSize = getNodeSize(n);
                baseSizes[n.id] = baseSize;
                sizeMap[n.id] = baseSize * currentZoomLevel;
            }});

            // Rebuild edges
            const edgeSegments = {{
                alert: {{ x: [], y: [] }},
                investigation: {{ x: [], y: [] }},
                normal: {{ x: [], y: [] }}
            }};

            edges.forEach(e => {{
                const s = e.source || e.from;
                const t = e.target || e.to;
                if (positions[s] && positions[t]) {{
                    let x0 = positions[s][0], y0 = positions[s][1];
                    let x1 = positions[t][0], y1 = positions[t][1];
                    
                    const startOffset = (sizeMap[s] || 75) / 2 + 6;
                    const endOffset = (sizeMap[t] || 75) / 2 + 6;
                    const coords = offsetEdge(x0, y0, x1, y1, startOffset, endOffset);
                    
                    const flags = e.flags || {{}};
                    let key = 'normal';
                    if (flags.alerted) key = 'alert';
                    else if (flags.investigated) key = 'investigation';
                    
                    edgeSegments[key].x.push(coords[0], coords[2], null);
                    edgeSegments[key].y.push(coords[1], coords[3], null);
                }}
            }});

            // Rebuild nodes
            const nodeX = [], nodeY = [], ringX = [], ringY = [];
            const nodeSizes = [], ringSizes = [], nodeLineWidths = [];

            nodes.forEach(n => {{
                const nid = n.id;
                if (positions[nid]) {{
                    const x = positions[nid][0], y = positions[nid][1];
                    nodeX.push(x);
                    nodeY.push(y);
                    nodeSizes.push(sizeMap[nid]);
                    
                    const isCenter = nid === rootNodeId;
                    const baseLineWidth = isCenter ? 3 : 2;
                    nodeLineWidths.push(baseLineWidth * currentZoomLevel);

                    const flags = n.flags || {{}};
                    if (flags.investigated) {{
                        ringX.push(x);
                        ringY.push(y);
                        ringSizes.push(sizeMap[nid] + 12 * currentZoomLevel);
                    }}
                }}
            }});

            // Update all traces with positions and sizes
            Plotly.restyle(graphDiv, {{
                'x': [edgeSegments.normal.x, edgeSegments.alert.x, edgeSegments.investigation.x, ringX, nodeX],
                'y': [edgeSegments.normal.y, edgeSegments.alert.y, edgeSegments.investigation.y, ringY, nodeY],
                'marker.size': [undefined, undefined, undefined, ringSizes, nodeSizes],
                'line.width': [2 * currentZoomLevel, 2 * currentZoomLevel, 2 * currentZoomLevel, undefined, undefined],
                'marker.line.width': [undefined, undefined, undefined, 3 * currentZoomLevel, nodeLineWidths],
                'textfont.size': [undefined, undefined, undefined, undefined, 9 * currentZoomLevel]
            }}, [0, 1, 2, 3, 4]);

            // Update axis ranges to fit the zoomed graph
            const dynamicRange = baseRange * Math.max(1, currentZoomLevel);
            Plotly.relayout(graphDiv, {{
                'xaxis.range': [-dynamicRange, dynamicRange],
                'yaxis.range': [-dynamicRange, dynamicRange]
            }});
        }}

        function zoomIn() {{
            const newZoom = currentZoomLevel * 1.3;
            if (newZoom <= maxZoom) {{
                currentZoomLevel = newZoom;
                rebuildGraph();
                updateZoomButtons();
            }}
        }}

        function zoomOut() {{
            const newZoom = currentZoomLevel / 1.3;
            if (newZoom >= minZoom) {{
                currentZoomLevel = newZoom;
                rebuildGraph();
                updateZoomButtons();
            }}
        }}

        function resetZoom() {{
            currentZoomLevel = 1.0;
            rebuildGraph();
            updateZoomButtons();
        }}

        function updateZoomButtons() {{
            const zoomInBtn = document.getElementById('zoom-in-btn');
            const zoomOutBtn = document.getElementById('zoom-out-btn');
            const resetBtn = document.getElementById('reset-zoom-btn');
            
            if (currentZoomLevel * 1.3 > maxZoom) {{
                zoomInBtn.style.opacity = '0.4';
                zoomInBtn.style.cursor = 'not-allowed';
            }} else {{
                zoomInBtn.style.opacity = '1';
                zoomInBtn.style.cursor = 'pointer';
            }}
            
            if (currentZoomLevel / 1.3 < minZoom) {{
                zoomOutBtn.style.opacity = '0.4';
                zoomOutBtn.style.cursor = 'not-allowed';
            }} else {{
                zoomOutBtn.style.opacity = '1';
                zoomOutBtn.style.cursor = 'pointer';
            }}
            
            const zoomPercent = Math.round(currentZoomLevel * 100);
            resetBtn.setAttribute('title', 'Reset Zoom (current: ' + zoomPercent + '%)');
        }}

        function updateSidebar(nodeId) {{
            selectedNodeId = nodeId;
            const node = nodes.find(n => n.id === nodeId) || {{}};
            const isCenter = nodeId === rootNodeId;
            const stats = statsForNode(nodeId);

            const accountId = isCenter ? (accountDetails.accountId || node.id) : node.id;
            const accountHolder = isCenter ? (accountDetails.accountHolder || 'Unknown') : 'Unknown';
            const relationship = isCenter ? (accountDetails.relationship || accountDetails.accountRelationship || accountDetails.relationshipType) : (node.relationship || node.accountRelationship || node.relationshipType);
            const velocity = isCenter ? (accountDetails.velocity || node.velocity) : (node.velocity || '-');

            document.getElementById('sidebar-account-id').innerText = formatDisplayId(accountId);
            document.getElementById('sidebar-account-holder').innerText = accountHolder || 'Unknown';
            document.getElementById('sidebar-relationship').innerText = relationship || '-';

            document.getElementById('stat-total-tx').innerText = numberFormatter.format(stats.tx || 0);
            document.getElementById('stat-total-value').innerText = currencyFormatter.format(stats.total || 0);
            document.getElementById('stat-velocity').innerText = velocity || '-';

            // Use flags structure for alerts and investigations
            const accountFlags = accountDetails.flags || {{}};
            const nodeFlags = node.flags || {{}};
            const isAlert = isCenter ? !!accountFlags.alerted : !!nodeFlags.alerted;
            const isInvestigation = isCenter ? !!accountFlags.investigated : !!nodeFlags.investigated;

            document.getElementById('alert-box').style.display = isAlert ? 'block' : 'none';
            document.getElementById('investigation-box').style.display = isInvestigation ? 'block' : 'none';
        }}

        // Add mousewheel zoom support
        if (graphDiv) {{
            graphDiv.addEventListener('wheel', function(e) {{
                e.preventDefault();
                if (e.deltaY < 0) {{
                    zoomIn();
                }} else {{
                    zoomOut();
                }}
            }}, {{ passive: false }});

            graphDiv.on('plotly_click', d => {{
                if (d.points && d.points[0] && d.points[0].customdata) {{
                    const nodeId = d.points[0].customdata;
                    updateSidebar(nodeId);
                }}
            }});
        }}

        document.getElementById('zoom-in-btn').addEventListener('click', zoomIn);
        document.getElementById('zoom-out-btn').addEventListener('click', zoomOut);
        document.getElementById('reset-zoom-btn').addEventListener('click', resetZoom);
        
        updateZoomButtons();
        
        if (selectedNodeId) {{
            updateSidebar(selectedNodeId);
        }}
    }})();
    </script>
    """

    container_html = f"""
    <div style="display: flex; background: #ffffff; border: 1px solid #e2e8f0; border-radius: 12px; overflow: hidden; box-shadow: 0 6px 16px rgba(15,23,42,0.08);">
        <div style="flex: 1; background: white; position: relative;">
            {graph_html}
            {legend_html}
            {zoom_controls_html}
        </div>
        <div style="width: 1px; background: #e2e8f0;"></div>
        {sidebar_html}
    </div>
    {interactive_script}
    """

    display(HTML(container_html))
else:
    display(HTML('<div style="color: #ef4444; padding: 20px;">No network data available</div>'))